# PDF to Text — OCR Pipeline
**Support:** PDF Digital + PDF Scan/Foto | Bahasa: Indonesia & Inggris  
**Model OCR:** EasyOCR (Deep Learning) | **PDF Parser:** pdfplumber + PyMuPDF

---
###  Alur Pipeline
```
PDF Input
   │
   ├─► Cek tipe PDF
   │       │
   │       ├─ Digital (ada text layer) → pdfplumber extraction
   │       │
   │       └─ Scan/Foto (tidak ada text) → EasyOCR (Deep Learning)
   │
   └─► Output: teks bersih per halaman + export .txt / .json
```


## 1️ Install Dependencies

In [24]:
!pip install easyocr pdfplumber PyMuPDF Pillow tqdm --quiet

print(" Semua library berhasil diinstall!")


 Semua library berhasil diinstall!


## 2️ Import Library

In [25]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import fitz
import pdfplumber
import easyocr
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

print(" Semua library berhasil diimport!")
print(f"   EasyOCR version : {easyocr.__version__}")
print(f"   PyMuPDF version : {fitz.__version__}")


 Semua library berhasil diimport!
   EasyOCR version : 1.7.2
   PyMuPDF version : 1.27.2.2


## 3️ Konfigurasi

In [26]:
PDF_PATH = "/content/Profile.pdf"
OCR_LANGUAGES = ['id', 'en']
MIN_CHARS_DIGITAL = 30
RENDER_DPI = 200


OUTPUT_TXT  = "output_teks.txt"
OUTPUT_JSON = "output_data.json"
USE_GPU = False

print(" Konfigurasi siap!")
print(f"   PDF        : {PDF_PATH}")
print(f"   Bahasa OCR : {OCR_LANGUAGES}")
print(f"   Render DPI : {RENDER_DPI}")
print(f"   GPU        : {USE_GPU}")


 Konfigurasi siap!
   PDF        : /content/Profile.pdf
   Bahasa OCR : ['id', 'en']
   Render DPI : 200
   GPU        : False


## 4️ Inisialisasi EasyOCR
> Proses ini mendownload model Deep Learning (~100MB) hanya sekali.  
> Model yang digunakan: **CRAFT** (text detection) + **CRNN** (text recognition)


In [27]:
print(" Memuat model EasyOCR...")
print("   (Download model DL pertama kali ~100MB, tunggu sebentar...)")

reader = easyocr.Reader(
    OCR_LANGUAGES,
    gpu=USE_GPU,
    verbose=False
)

print(" EasyOCR siap digunakan!")
print(f"   Bahasa aktif: {OCR_LANGUAGES}")


 Memuat model EasyOCR...
   (Download model DL pertama kali ~100MB, tunggu sebentar...)
 EasyOCR siap digunakan!
   Bahasa aktif: ['id', 'en']


## 5️⃣ Fungsi Utama

In [28]:
def is_digital_page(page_plumber, min_chars: int = MIN_CHARS_DIGITAL) -> bool:
    text = page_plumber.extract_text() or ""
    return len(text.strip()) >= min_chars


def extract_text_digital(page_plumber) -> str:
    text = page_plumber.extract_text()
    if text:
        return text.strip()
    return ""


def page_to_image(pdf_doc_fitz, page_num: int, dpi: int = RENDER_DPI) -> np.ndarray:
    page = pdf_doc_fitz[page_num]
    zoom  = dpi / 72
    mat   = fitz.Matrix(zoom, zoom)
    pix   = page.get_pixmap(matrix=mat, alpha=False)
    img   = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return np.array(img)


def extract_text_ocr(image_array: np.ndarray, ocr_reader) -> str:
    results = ocr_reader.readtext(
        image_array,
        detail=0,
        paragraph=True
    )
    return "\n".join(results).strip()


def extract_text_ocr_with_confidence(image_array: np.ndarray, ocr_reader,
                                      confidence_threshold: float = 0.3):
    results = ocr_reader.readtext(image_array, detail=1, paragraph=False)
    filtered = [(bbox, text, conf) for (bbox, text, conf) in results
                if conf >= confidence_threshold]

    if not filtered:
        return "", 0.0

    texts      = [text for (_, text, _) in filtered]
    avg_conf   = sum(conf for (_, _, conf) in filtered) / len(filtered)
    full_text  = " ".join(texts)
    return full_text.strip(), round(avg_conf, 4)


print(" Semua fungsi siap!")


 Semua fungsi siap!


## 6️ Jalankan Pipeline PDF → Text

In [29]:
def process_pdf(pdf_path: str) -> list[dict]:
    """
    Pipeline utama:
    - Auto-detect tiap halaman: digital atau scan
    - Digital → pdfplumber
    - Scan    → EasyOCR (Deep Learning)

    Returns list of dict per halaman.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f" File tidak ditemukan: {pdf_path}")

    results     = []
    start_total = time.time()

    # Buka PDF dengan dua library sekaligus
    pdf_fitz    = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)
    total_pages = len(pdf_fitz)

    print(f" File   : {pdf_path}")
    print(f" Halaman: {total_pages}")
    print(f"{'─'*50}")

    stat_digital = 0
    stat_ocr     = 0

    for page_num in tqdm(range(total_pages), desc=" Memproses halaman"):
        t_start = time.time()
        page_info = {
            "halaman"    : page_num + 1,
            "tipe"       : "",
            "teks"       : "",
            "confidence" : None,
            "waktu_detik": 0.0
        }

        plumber_page = pdf_plumber.pages[page_num]

        if is_digital_page(plumber_page):
            # ── Mode Digital ──────────────────────────────────
            page_info["tipe"] = "digital"
            page_info["teks"] = extract_text_digital(plumber_page)
            stat_digital += 1
        else:
            # ── Mode OCR (Scan/Foto) ──────────────────────────
            page_info["tipe"] = "ocr"
            img_array         = page_to_image(pdf_fitz, page_num, dpi=RENDER_DPI)
            text, conf        = extract_text_ocr_with_confidence(img_array, reader)
            page_info["teks"]       = text
            page_info["confidence"] = conf
            stat_ocr += 1

        page_info["waktu_detik"] = round(time.time() - t_start, 2)
        results.append(page_info)

    pdf_fitz.close()
    pdf_plumber.close()

    total_time = round(time.time() - start_total, 2)

    print(f"{'─'*50}")
    print(f" Selesai dalam {total_time} detik")
    print(f"    Halaman digital (pdfplumber): {stat_digital}")
    print(f"    Halaman scan/OCR (EasyOCR) : {stat_ocr}")

    return results


# ── Jalankan ──────────────────────────────────────────────────────────────────
hasil = process_pdf(PDF_PATH)


 File   : /content/Profile.pdf
 Halaman: 2
──────────────────────────────────────────────────


 Memproses halaman:   0%|          | 0/2 [00:00<?, ?it/s]

──────────────────────────────────────────────────
 Selesai dalam 0.13 detik
    Halaman digital (pdfplumber): 2
    Halaman scan/OCR (EasyOCR) : 0


## 7️⃣ Preview Hasil

In [30]:
def preview_hasil(results: list[dict], n_preview: int = 3):
    """Tampilkan preview teks dari beberapa halaman pertama."""
    print(f"{'═'*60}")
    print(f"  PREVIEW HASIL ({min(n_preview, len(results))} halaman pertama)")
    print(f"{'═'*60}")

    for r in results[:n_preview]:
        label_tipe = " Digital" if r["tipe"] == "digital" else " OCR Scan"
        conf_info  = f" | Confidence: {r['confidence']:.2%}" if r["confidence"] else ""
        print(f"\n{'─'*60}")
        print(f"Halaman {r['halaman']} | {label_tipe} | ⏱ {r['waktu_detik']}s{conf_info}")
        print(f"{'─'*60}")
        teks_preview = r["teks"][:500] + ("..." if len(r["teks"]) > 500 else "")
        print(teks_preview if teks_preview else "(kosong / tidak ada teks)")

    print(f"\n{'═'*60}")
    total_chars = sum(len(r["teks"]) for r in results)
    print(f"  Total karakter teks: {total_chars:,}")
    print(f"{'═'*60}")

preview_hasil(hasil, n_preview=3)


════════════════════════════════════════════════════════════
  PREVIEW HASIL (2 halaman pertama)
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
Halaman 1 |  Digital | ⏱ 0.1s
────────────────────────────────────────────────────────────
Contact Novita Anggraini
www.linkedin.com/in/novitanggraini Data enthusiast
(LinkedIn) Pontianak, West Kalimantan, Indonesia
Top Skills Summary
Data Validation
Open to opportunities in data analytics and software-related roles,
Kecerdasan Buatan (AI)
with interest in applying machine learning for practical solutions.
Data Processing
Currently growing skills in data analysis, visualization, and software
development while working on various personal and academic
Certifications
projects.
Artificial...

────────────────────────────────────────────────────────────
Halaman 2 |  Digital | ⏱ 0.02s
────────────────────────────────────────────────────────────
Bachelor of Computer Science, C

## 8️ statistik & Analisis

In [31]:
def tampilkan_statistik(results: list[dict]):
    total      = len(results)
    n_digital  = sum(1 for r in results if r["tipe"] == "digital")
    n_ocr      = sum(1 for r in results if r["tipe"] == "ocr")
    n_kosong   = sum(1 for r in results if not r["teks"].strip())

    ocr_pages  = [r for r in results if r["tipe"] == "ocr" and r["confidence"] is not None]
    avg_conf   = (sum(r["confidence"] for r in ocr_pages) / len(ocr_pages)) if ocr_pages else 0

    total_time = sum(r["waktu_detik"] for r in results)
    total_char = sum(len(r["teks"]) for r in results)
    total_word = sum(len(r["teks"].split()) for r in results)

    print(f"{'═'*55}")
    print(f"   STATISTIK HASIL EKSTRAKSI")
    print(f"{'═'*55}")
    print(f"  Total halaman          : {total}")
    print(f"  ├─ Halaman digital     : {n_digital}  ({n_digital/total*100:.1f}%)")
    print(f"  ├─ Halaman scan (OCR)  : {n_ocr}  ({n_ocr/total*100:.1f}%)")
    print(f"  └─ Halaman kosong      : {n_kosong}")
    print(f"{'─'*55}")
    print(f"  Total karakter         : {total_char:,}")
    print(f"  Total kata (estimasi)  : {total_word:,}")
    if ocr_pages:
        print(f"  Rata-rata confidence   : {avg_conf:.2%}")
    print(f"  Total waktu proses     : {total_time:.2f} detik")
    print(f"  Rata-rata per halaman  : {total_time/total:.2f} detik")
    print(f"{'═'*55}")

    # Halaman dengan confidence rendah (< 50%)
    low_conf = [r for r in ocr_pages if r["confidence"] < 0.5]
    if low_conf:
        print(f"\n  Halaman OCR dengan confidence rendah (<50%):")
        for r in low_conf:
            print(f"   - Halaman {r['halaman']} : {r['confidence']:.2%}")

tampilkan_statistik(hasil)


═══════════════════════════════════════════════════════
   STATISTIK HASIL EKSTRAKSI
═══════════════════════════════════════════════════════
  Total halaman          : 2
  ├─ Halaman digital     : 2  (100.0%)
  ├─ Halaman scan (OCR)  : 0  (0.0%)
  └─ Halaman kosong      : 0
───────────────────────────────────────────────────────
  Total karakter         : 1,592
  Total kata (estimasi)  : 207
  Total waktu proses     : 0.12 detik
  Rata-rata per halaman  : 0.06 detik
═══════════════════════════════════════════════════════


## 9️ Simpan Output

In [32]:
def simpan_output(results: list[dict],
                  txt_path: str = OUTPUT_TXT,
                  json_path: str = OUTPUT_JSON):
    """Simpan hasil ke file .txt dan .json"""

    # ── Simpan TXT ────────────────────────────────────────────────────────────
    with open(txt_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(f"\n{'='*60}\n")
            f.write(f"HALAMAN {r['halaman']} | Tipe: {r['tipe'].upper()}\n")
            f.write(f"{'='*60}\n")
            f.write(r["teks"] + "\n")
    print(f" Teks plain tersimpan  : {txt_path}")

    # ── Simpan JSON ───────────────────────────────────────────────────────────
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({
            "file_sumber" : PDF_PATH,
            "total_halaman": len(results),
            "halaman"     : results
        }, f, ensure_ascii=False, indent=2)
    print(f" Data JSON tersimpan   : {json_path}")

    # ── Info ukuran file ──────────────────────────────────────────────────────
    size_txt  = os.path.getsize(txt_path) / 1024
    size_json = os.path.getsize(json_path) / 1024
    print(f"   {txt_path}  → {size_txt:.1f} KB")
    print(f"   {json_path} → {size_json:.1f} KB")


simpan_output(hasil)


 Teks plain tersimpan  : output_teks.txt
 Data JSON tersimpan   : output_data.json
   output_teks.txt  → 1.9 KB
   output_data.json → 1.9 KB


## 10 Utilitas Tambahan

In [33]:
# ── Cari teks tertentu di seluruh dokumen ────────────────────────────────────
def cari_teks(results: list[dict], kata_kunci: str) -> None:
    """Cari kata kunci di semua halaman (case-insensitive)."""
    kata = kata_kunci.lower()
    ditemukan = []
    for r in results:
        if kata in r["teks"].lower():
            idx   = r["teks"].lower().find(kata)
            start = max(0, idx - 80)
            end   = min(len(r["teks"]), idx + 80)
            cuplikan = "..." + r["teks"][start:end].replace("\n", " ") + "..."
            ditemukan.append((r["halaman"], cuplikan))

    if ditemukan:
        print(f" Kata '{kata_kunci}' ditemukan di {len(ditemukan)} halaman:")
        for hal, cuplikan in ditemukan:
            print(f"\n   Halaman {hal}:")
            print(f"     {cuplikan}")
    else:
        print(f" Kata '{kata_kunci}' tidak ditemukan.")


# ── Ekstrak halaman tertentu saja ─────────────────────────────────────────────
def ambil_halaman(results: list[dict], nomor_halaman: int) -> str:
    """Ambil teks dari halaman tertentu (nomor dimulai dari 1)."""
    for r in results:
        if r["halaman"] == nomor_halaman:
            return r["teks"]
    return f"Halaman {nomor_halaman} tidak ditemukan."


# ── Contoh penggunaan ─────────────────────────────────────────────────────────
print(" Contoh fungsi utilitas:")
print()
print("# Cari kata tertentu:")
print("  cari_teks(hasil, 'kontrak')")
print()
print("# Ambil teks halaman tertentu:")
print("  teks_hal3 = ambil_halaman(hasil, 3)")
print("  print(teks_hal3)")
print()

# Uncomment untuk mencoba:
# cari_teks(hasil, "kata yang dicari")
# print(ambil_halaman(hasil, 1))


 Contoh fungsi utilitas:

# Cari kata tertentu:
  cari_teks(hasil, 'kontrak')

# Ambil teks halaman tertentu:
  teks_hal3 = ambil_halaman(hasil, 3)
  print(teks_hal3)



---

##  Selesai!

| Output | Keterangan |
|--------|-----------|
| `output_teks.txt` | Teks plain semua halaman |
| `output_data.json` | Data lengkap per halaman (teks + metadata) |

###  Tips Meningkatkan Akurasi OCR
- **DPI lebih tinggi** → Naikkan `RENDER_DPI` ke `250–300` untuk dokumen kecil/rapat
- **Gambar buram** → Tambahkan preprocessing (grayscale, thresholding) sebelum OCR  
- **Confidence rendah** → Periksa kualitas scan asli atau naikkan DPI
- **GPU tersedia** → Set `USE_GPU = True` untuk 5–10x lebih cepat

###  Library yang Digunakan
| Library | Fungsi | Model |
|---------|--------|-------|
| **EasyOCR** | OCR scan/foto | CRAFT (detection) + CRNN (recognition) |
| **pdfplumber** | Ekstrak teks PDF digital | Rule-based |
| **PyMuPDF** | Render halaman → gambar | Rendering engine |
